In [0]:
%run ../common/shipp_silver_lib

In [0]:
import uuid
from datetime import datetime, timezone

RUN_ORS = True
STAGES = [
    ("10_silver_listings", "../development/10_silver_listings_dev"),
    ("11_silver_requests", "../development/11_silver_requests_dev"),
    ("30_candidate_pairs", "../development/30_silver_candidate_pairs"),
    ("31_ors_enrichment", "../development/31_ors_route_enrichment"),
    ("32_silver_routes", "../development/32_silver_routes"),
    ("40_gold_matches", "../development/40_gold_candidate_matches"),
    ("60_marketplace_metrics", "../development/60_gold_marketplace_metrics"),
]
if not RUN_ORS:
    STAGES = [s for s in STAGES if s[0] != "31_ors_enrichment"]

LOG_SCHEMA = ("run_id string, stage string, started_at timestamp, ended_at timestamp, "
              "seconds double, status string, summary string")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {PIPELINE_RUN_LOG} ({LOG_SCHEMA}) USING DELTA")
RUN_ID = str(uuid.uuid4())
print("RUN_ID =", RUN_ID)

In [0]:
# SHIPP P0 — Read-only UC Volume image verification

volume_path = (
    "/Volumes/bootcamp_students/shipp_bronze/"
    "listing_images/"
    "a2c97986-ab61-47b4-b577-b28d90271ee8/"
)

expected_files = {
    "354a0932-7c4e-4fb6-a623-fdee6c4eb13c_Screenshot_2026-10-09_at_6.14.41_PM.png": 1745404,
    "5d579a89-6497-48b3-9c0a-2c454d3d2499_Screenshot_2026-10-09_at_6.15.02_PM.png": 1988219,
    "81d0f601-2ae4-4d20-bff5-d5a06bc5069c_Screenshot_2026-10-09_at_6.14.24_PM.png": 656389,
}

files = {
    file.name: file.size
    for file in dbutils.fs.ls(volume_path)
    if not file.isDir()
}

passed = 0

for name, expected_size in expected_files.items():
    actual_size = files.get(name)
    valid = actual_size == expected_size

    print(
        "PASS" if valid else "FAIL",
        "|", name,
        "| expected:", expected_size,
        "| actual:", actual_size
    )

    passed += int(valid)

print(f"\nVALIDATION: {passed}/3 files matched")

if passed == 3:
    print("PASS — UC Volume files verified")
else:
    print("FAIL — Investigate missing or mismatched files")

In [0]:

image_path = (
    "/Volumes/bootcamp_students/shipp_bronze/listing_images/"
    "a2c97986-ab61-47b4-b577-b28d90271ee8/"
    "354a0932-7c4e-4fb6-a623-fdee6c4eb13c_"
    "Screenshot_2026-10-09_at_6.14.41_PM.png"
)

with open(image_path, "rb") as f:
    header = f.read(16)

print("Image header:", header)
assert header.startswith(b"\x89PNG\r\n\x1a\n")
print("PASS: PNG image is readable")


In [0]:

image_path = (
    "/Volumes/bootcamp_students/shipp_bronze/listing_images/"
    "PASTE_LISTING_ID/PASTE_REAL_IMAGE_FILENAME"
)

with open(image_path, "rb") as f:
    header = f.read(16)

assert header, "Image file is empty"
print("PASS: Image object can be read")
FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/bootcamp_students/shipp_bronze/listing_images/PASTE_LISTING_ID/PASTE_REAL_IMAGE_FILENAME'
[Trace ID: 00-923a6b15d7b649d7552c489c455e19ff-12222e4a6a1f514b-00]
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-8401650290911633>, line 7
      2 image_path = (
      3     "/Volumes/bootcamp_students/shipp_bronze/listing_images/"
      4     "PASTE_LISTING_ID/PASTE_REAL_IMAGE_FILENAME"
      5 )
----> 7 with open(image_path, "rb") as f:
      8     header = f.read(16)
     10 assert header, "Image file is empty"

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:343, in _modified_open(file, *args, **kwargs)
    336 if file in {0, 1, 2}:
    337     raise ValueError(
    338         f"IPython won't let you open fd={file} by default "